# SPU demo3
干掉耗时的topk采样
## (1) Load Model & Weights from HuggingFace

In [1]:
import jax

from flax_rnn.helper import load_from_cache, generate, generate_topk, generate_greedy, generate_minp

base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


In [2]:
def generate_demo(prompt, gen_len=10, seed=42):
    input_ids = tokenizer.encode(prompt, return_tensors='jax')
    output_ids = generate_minp(model, params, input_ids, gen_len, seed=seed)
    print(prompt, tokenizer.decode(output_ids[0]), sep='')

In [3]:
generate_demo('Python is')

Python is in a bad state, and the fact that you


## (2) SPU

### 2.1: 定义simulator

> 教程视频中用了一堆的`spu_pb2`，但现在的spu中根本没有这个模块，根据成员推测，尝试换成`libspu`

In [4]:
import spu.utils.simulation as spsim
import spu.libspu as libspu

# sim_che = spsim.Simulator.simple(2,libspu.ProtocolKind.CHEETAH, libspu.FieldType.FM128)
# sim_aby = spsim.Simulator.simple(3,libspu.ProtocolKind.ABY3, libspu.FieldType.FM128)

下面的参数来自flax_resnet示例，精度很高，输出很理想，但可能运行会慢一些

我们特意让aby的参数和`3pc.json`一致

In [5]:
# define cheetah config with pphlo trace and profile on
config_che = libspu.RuntimeConfig(
    protocol=libspu.ProtocolKind.CHEETAH,
    field=libspu.FieldType.FM128,
    fxp_fraction_bits=36,
)

config_che.enable_pphlo_profile = True
config_che.enable_hal_profile = True

config_aby = libspu.RuntimeConfig(
    protocol=libspu.ProtocolKind.ABY3,
    field=libspu.FieldType.FM128,
    fxp_fraction_bits=36,
)

config_aby.enable_pphlo_profile = True
config_aby.enable_hal_profile = True
config_aby.fxp_exp_mode = libspu.RuntimeConfig.ExpMode.EXP_PADE
config_aby.fxp_div_goldschmidt_iters = 3

sim = spsim.Simulator(3, config_aby)

### 2.2: 定义运行函数

In [6]:
# 目的是加上jit
# 重要：topk非常非常慢，而且运行时间和topk的值线性相关
@jax.jit
def gen_spu(params, input_ids):
    # return generate(model, params, input_ids, n_tokens_to_gen=3) # 老代码
    # return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快
    # return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢
    return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

### 2.3: 运行密态程序

In [7]:
# plaintext test
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
output_ids = gen_spu(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is in a bad


In [8]:
# SPU emulation mode
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
output_ids = spsim.sim_jax(sim, gen_spu)(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

[2026-04-07 14:38:54.177] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 14:41:42.878] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.106e-05s, execution took 133.429735343s, output processing took 5.44e-06s, total time 133.429801843s.
[2026-04-07 14:41:42.989] [info] [api.cc:220] HLO profiling: total time 133.23674436000002
[2026-04-07 14:41:42.989] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 41.374322562s, send bytes 4010041344 recv bytes 3753228288, send actions 22847, recv actions 21341
[2026-04-07 14:41:42.989] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 27.240138468s, send bytes 7667712 recv bytes 7913472, send actions 413, recv actions 424
[2026-04-07 14:41:42.989] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 25.39256153s, send bytes 25712256 recv bytes 26272064, send actions 800, recv actions 805
[2026-04-07 14:41:42.989] [info] [api.cc:2

2′20″后，
```plaintext
Python is a great tool
```

## (3) SPU下验证性能
### 3.1 定义emulator

In [9]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS

# note: in MULTIPROCESS mode, bandwidth and latency doesn't work
# emulation.CLUSTER_ABY3_3PC is a hard-coded string
# we copied it to current folder
emulator = emulation.Emulator(
    "3pc.json",
    mode,
    bandwidth=100,
    latency=10
)

emulator.up()

[2026-04-07 14:41:45,078]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-07 14:41:45,660] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-07 14:41:45,660] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-07 14:41:45,662] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-07 14:41:45,667] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-07 14:41:45,672] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-07 14:41:47,156] [ForkServerProcess-1] Run : builtin_spu_init at node:0
[2026-04-07 14:41:47,156] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-07 14:41:47,156] [ForkServerProcess-3] Run : builtin_spu_init at node:2
I0407 14:41:47.164182 1047512     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61930.
W0407 14:41:47.164205 1047512     0 external/brpc~/src/brpc/server.cpp

In [10]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
[2026-04-07 14:41:48,584] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 14:41:48,641] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 14:41:48,644] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-07 14:41:48.645] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-07 14:41:55,144] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 14:41:55,149] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 14:41:55,155] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 14:41:55,158] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 14:41:55,159] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 14:41:55,162] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 14:41:55,163] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 14:41:55,165] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 14:41:55,167] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 14:41:55,170] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 14:41:55,171] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 14:41:55,173] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 14:41:55,181] [ForkServerProcess-4

[2026-04-07 14:42:32.196] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 14:42:32.270] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 14:42:32.582] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 14:44:44.572] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.6909e-05s, execution took 132.050411546s, output processing took 3.03e-06s, total time 132.050481485s.
[2026-04-07 14:44:44.638] [info] [api.cc:220] HLO profiling: total time 131.95941262
[2026-04-07 14:44:44.638] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 42.016162944s, send bytes 4032430080 recv bytes 3773620224, send actions 22879, recv actions 21310
[2026-04-07 14:44:44.638] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 26.720902784s, send bytes 7569408 recv bytes 7606272, send actions 410, recv actions 413
[2026-04-07 14:44:44.638] [info] [api.c

[2026-04-07 14:44:44,896] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-07 14:44:44,902] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 14:44:44,902] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 14:44:44,902] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 14:44:44,903] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 14:44:44,904] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 14:44:44,952] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 14:44:44,953] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 14:44:44,953] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 14:44:44,953] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 14:44:44,953] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 14:44:44,976] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 14:44:44,977] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 14:44

干掉topk之后速度非常快，现在耗时的大头在exp
```plaintext
[2026-04-02 16:33:20.142] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.602e-05s, execution took 131.807348882s, output processing took 4.22e-06s, total time 131.807419122s.
[2026-04-02 16:33:20.206] [info] [api.cc:220] HLO profiling: total time 131.72138209299996
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 42.315413413s, send bytes 4016873472 recv bytes 3750727680, send actions 22888, recv actions 21305
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 26.410895758s, send bytes 7643136 recv bytes 7839744, send actions 411, recv actions 418
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 22.208245624s, send bytes 81007616 recv bytes 80048384, send actions 98440, recv actions 98528
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 17.639317426s, send bytes 23644800 recv bytes 22218560, send actions 784, recv actions 763
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.multiply, executed 951 times, duration 6.549204685s, send bytes 282429376 recv bytes 279148616, send actions 2428, recv actions 2314
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.log_plus_one, executed 72 times, duration 3.575090644s, send bytes 380436480 recv bytes 356241408, send actions 3093, recv actions 3155
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.reciprocal, executed 144 times, duration 3.547714335s, send bytes 304594944 recv bytes 309018624, send actions 8608, recv actions 8755
```
换成min_p采样之后，运行时间和贪心几乎一致，但是采样效果好多了
```plaintext
[2026-04-07 11:47:42.667] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.596e-05s, execution took 135.07210997s, output processing took 3.94e-06s, total time 135.07217987s.
[2026-04-07 11:47:42.708] [info] [api.cc:220] HLO profiling: total time 134.97185146800004
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 43.338933034s, send bytes 4031643648 recv bytes 3747151872, send actions 22962, recv actions 21450
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 26.853463713s, send bytes 7569408 recv bytes 7643136, send actions 409, recv actions 410
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 21.98340975s, send bytes 82087936 recv bytes 82017792, send actions 98249, recv actions 97975
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 18.335805278s, send bytes 24136704 recv bytes 23051776, send actions 792, recv actions 801
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.multiply, executed 951 times, duration 7.711868615s, send bytes 281753472 recv bytes 279547784, send actions 2405, recv actions 2252
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.log_plus_one, executed 72 times, duration 3.557931869s, send bytes 368664576 recv bytes 388276224, send actions 3060, recv actions 3131
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.reciprocal, executed 144 times, duration 3.55727766s, send bytes 303071232 recv bytes 307458048, send actions 8571, recv actions 8711
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.pad, executed 49 times, duration 3.073966639s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-07 11:47:42.708] [info] [api.cc:223] - pphlo.reduce, executed 131 times, duration 1.948924919s, send bytes 96910311 recv bytes 87048487, send actions 3279, recv actions 3155
```

In [11]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


In [12]:
# emulator.down()